**categorical descriptive statistics**

In [ ]:

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/kaggle/input/datasets/at13lemmewrite/zomato-dataset-csv/Zomato Dataset.csv")
# Check first five rows
print(df.head())

# Check number of rows and columns
print(df.shape)

# Check column names and data types
print(df.info())

# Check missing values
print(df.isnull().sum())

# Check duplicate records
print(df.duplicated().sum())

# Descriptive statistics
print(df.describe())


In [ ]:

# Descriptive statistics for numerical variables

numeric_data = df.select_dtypes(include=np.number)

print("Numerical Descriptive Statistics:")
print(numeric_data.describe().T)


In [ ]:

# Descriptive statistics for categorical variables

categorical_data = df.select_dtypes(include='object')

print("Categorical Descriptive Statistics:")
print(categorical_data.describe().T)


In [ ]:

# Frequency distribution of important variables

print("Traffic Density:")
print(df['Road_traffic_density'].value_counts())

print("\nWeather Conditions:")
print(df['Weather_conditions'].value_counts())

print("\nVehicle Type:")
print(df['Type_of_vehicle'].value_counts())

print("\nCity:")
print(df['City'].value_counts())

print("\nFestival:")
print(df['Festival'].value_counts())


My observation for the same :

Vehicle concentration: Motorcycles account for approximately 58% of all delivery records. This makes vehicle type an important variable to investigate.

Traffic distribution: Low traffic and Jam together account for 29,615 records. We can compare their average delivery times later.

Festival imbalance: Only 896 records are festival deliveries. This imbalance must be considered when interpreting festival-related results.

Exploratory Data Analysis (EDA)

In [ ]:

# Missing value analysis

missing_values = df.isnull().sum()

print("Missing Values in Each Column:")
print(missing_values)

print("\nTotal Missing Values:")
print(missing_values.sum())


In [ ]:

# Percentage of missing values

missing_percentage = (df.isnull().sum() / len(df)) * 100

print("Missing Value Percentage:")
print(missing_percentage)


In [ ]:

# Combine missing counts and percentages

missing_summary = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage': missing_percentage
})

print(missing_summary[missing_summary['Missing Values'] > 0])


my observation
- Delivery ratings and age have the highest missing percentages.
- Order time is also missing in 3.8% of records.
- Festival has very few missing observations.
- Weather and traffic have relatively low missing percentages.

Outlier detection using the IQR method

In [ ]:

# Inspect important numerical variables

columns_to_check = [
    'Delivery_person_Age',
    'Delivery_person_Ratings',
    'multiple_deliveries',
    'Time_taken (min)'
]

for col in columns_to_check:
    print("\nVariable:", col)
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())
    print("Unique values:", df[col].nunique())
    print("Quartiles:")
    print(df[col].quantile([0.25, 0.50, 0.75]))


In [ ]:
#box plot ji
import matplotlib.pyplot as plt
import seaborn as sns

for col in columns_to_check:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[col])
    plt.title("Box Plot of " + col)
    plt.xlabel(col)
    plt.show()


investigate the rating value of 6

In [ ]:
print("Rating values:")
print(sorted(df['Delivery_person_Ratings'].dropna().unique()))

print("\nRecords with rating above 5:")
print(df[df['Delivery_person_Ratings'] > 5][
    ['Delivery_person_ID', 'Delivery_person_Ratings']
].head(10))

my observation : dataset contains ratings of 6.0, while all the other observed ratings are between 1.0 and 5.0.

In [ ]:
#Step 1: Count how many records have a rating of 6
print("Number of ratings equal to 6:")
print((df['Delivery_person_Ratings'] == 6).sum())

print("\nPercentage of ratings equal to 6:")
print((df['Delivery_person_Ratings'] == 6).mean() * 100)

check the age variable

In [ ]:
print("Age values:")
print(sorted(df['Delivery_person_Age'].dropna().unique()))

print("\nRecords with age below 18:")
print(df[df['Delivery_person_Age'] < 18][
    ['Delivery_person_ID', 'Delivery_person_Age']
])

print("\nRecords with age above 45:")
print(df[df['Delivery_person_Age'] > 45][
    ['Delivery_person_ID', 'Delivery_person_Age']
])

Age 15: 38 records. These need verification because they are unusually low for delivery personnel.

Age 50: 53 records. These are not automatically incorrect; age 50 can be a valid age.

Both values occur repeatedly, so we should investigate them rather than assume they are random errors.

In [ ]:
print("Records with both age 50 and rating 6:")
print(
    df[
        (df['Delivery_person_Age'] == 50) &
        (df['Delivery_person_Ratings'] == 6)
    ][
        ['Delivery_person_ID', 'Delivery_person_Age',
         'Delivery_person_Ratings', 'Time_taken (min)']
    ]
)

print("\nCount:")
print(
    (
        (df['Delivery_person_Age'] == 50) &
        (df['Delivery_person_Ratings'] == 6)
    ).sum()
)

Calculate outliers using IQR

In [ ]:
columns_to_check = [
    'Delivery_person_Age',
    'Delivery_person_Ratings',
    'multiple_deliveries',
    'Time_taken (min)'
]

for col in columns_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_limit) |
        (df[col] > upper_limit)
    ]

    print("\nVariable:", col)
    print("Lower limit:", lower_limit)
    print("Upper limit:", upper_limit)
    print("Number of outliers:", len(outliers))
    print("Percentage of outliers:", round(len(outliers) / df[col].notna().sum() * 100, 2), "%")

observation : The IQR method identified potential outliers in delivery-person ratings, multiple deliveries and delivery time. No IQR outliers were identified in delivery-person age. However, statistical outliers were not automatically treated as errors.

check missing values and duplicates again

In [ ]:
data_quality = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage': (df.isnull().sum() / len(df)) * 100,
    'Data Type': df.dtypes
})

print(data_quality)
print("\nTotal duplicate rows:", df.duplicated().sum())

Examine missing values

In [ ]:
columns_with_missing = df.columns[df.isnull().any()]

for col in columns_with_missing:
    print("\nVariable:", col)

    print("Missing records:", df[col].isnull().sum())

    print("Average delivery time when value is missing:")
    print(df.loc[df[col].isnull(), 'Time_taken (min)'].mean())

    print("Average delivery time when value is available:")
    print(df.loc[df[col].notnull(), 'Time_taken (min)'].mean())

Create a cleaned copy and handle missing values

In [ ]:
# Keep the original dataset unchanged
df_clean = df.copy()

# Numerical columns: median imputation + missing indicators
numeric_cols = [
    'Delivery_person_Age',
    'Delivery_person_Ratings',
    'multiple_deliveries'
]

for col in numeric_cols:
    df_clean[col + '_missing'] = df_clean[col].isnull().astype(int)
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical columns: fill missing values as Unknown
categorical_cols = [
    'Time_Orderd',
    'Weather_conditions',
    'Road_traffic_density',
    'Festival',
    'City'
]

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

# Verify missing values after treatment
print("Missing values after treatment:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

print("\nOriginal dataset shape:", df.shape)
print("Cleaned dataset shape:", df_clean.shape)

Check geographical coordinates

In [ ]:
coordinate_cols = [
    'Restaurant_latitude',
    'Restaurant_longitude',
    'Delivery_location_latitude',
    'Delivery_location_longitude'
]

print("Coordinate summary:")
print(df_clean[coordinate_cols].describe().T)

print("\nRecords with coordinates equal to 0.01:")
print(
    (df_clean[coordinate_cols] == 0.01).sum()
)

print("\nRecords with negative coordinates:")
print(
    (df_clean[coordinate_cols] < 0).sum()
)

print("\nSample records with suspicious coordinates:")
print(
    df_clean[
        (df_clean[coordinate_cols] == 0.01).any(axis=1) |
        (df_clean[coordinate_cols] < 0).any(axis=1)
    ][coordinate_cols].head(10)
)

observation:

The delivery coordinates contain 0.01 in both latitude and longitude for 327 records. These may be placeholder values rather than actual GPS locations.

Negative restaurant coordinates are suspicious because this dataset represents Indian delivery operations. The sample also contains restaurant coordinates of (0, 0), which need investigation.

Check exact zero and suspicious coordinate records

In [ ]:
# Count exact zero values in each coordinate column
print("Exact zero values:")
print((df_clean[coordinate_cols] == 0).sum())

# Count records with at least one zero coordinate
zero_records = df_clean[
    (df_clean[coordinate_cols] == 0).any(axis=1)
]

print("\nRecords with at least one zero coordinate:")
print(len(zero_records))

# Count records with negative restaurant coordinates
negative_records = df_clean[
    (df_clean['Restaurant_latitude'] < 0) |
    (df_clean['Restaurant_longitude'] < 0)
]

print("\nRecords with negative restaurant coordinates:")
print(len(negative_records))

# Count records with delivery coordinates equal to 0.01
placeholder_records = df_clean[
    (df_clean['Delivery_location_latitude'] == 0.01) &
    (df_clean['Delivery_location_longitude'] == 0.01)
]

print("\nRecords with both delivery coordinates equal to 0.01:")
print(len(placeholder_records))

Flag unreliable GPS records

In [ ]:
# Create a flag for unreliable GPS coordinates
df_clean['invalid_gps'] = (
    (df_clean['Restaurant_latitude'] <= 0) |
    (df_clean['Restaurant_longitude'] <= 0) |
    (
        (df_clean['Delivery_location_latitude'] == 0.01) &
        (df_clean['Delivery_location_longitude'] == 0.01)
    )
).astype(int)

# Check the combined count
print("Records with unreliable GPS coordinates:")
print(df_clean['invalid_gps'].sum())

print("\nRecords with valid GPS coordinates:")
print((df_clean['invalid_gps'] == 0).sum())

print("\nFlag distribution:")
print(df_clean['invalid_gps'].value_counts())

In [ ]:
#Calculate delivery distance in kilometres
# Create a valid GPS mask
valid_gps = df_clean['invalid_gps'] == 0

# Convert latitude and longitude into radians
lat1 = np.radians(df_clean['Restaurant_latitude'])
lon1 = np.radians(df_clean['Restaurant_longitude'])

lat2 = np.radians(df_clean['Delivery_location_latitude'])
lon2 = np.radians(df_clean['Delivery_location_longitude'])

# Calculate differences
dlat = lat2 - lat1
dlon = lon2 - lon1

# Haversine formula
a = (
    np.sin(dlat / 2) ** 2 +
    np.cos(lat1) * np.cos(lat2) *
    np.sin(dlon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))

# Distance in kilometres
df_clean['distance_km'] = np.where(
    valid_gps,
    6371 * c,
    np.nan
)

# Check the result
print("Distance summary:")
print(df_clean['distance_km'].describe())

print("\nMissing distance values:")
print(df_clean['distance_km'].isnull().sum())

print("\nSample calculated distances:")
print(df_clean[['distance_km', 'Time_taken (min)']].head(10))

Visualise delivery distance

In [ ]:
# Histogram of delivery distance
plt.figure(figsize=(8, 5))

sns.histplot(
    df_clean['distance_km'].dropna(),
    bins=30,
    kde=True
)

plt.title('Distribution of Food Delivery Distance')
plt.xlabel('Delivery Distance (km)')
plt.ylabel('Number of Orders')

plt.show()

In [ ]:
# Boxplot of delivery distance
plt.figure(figsize=(8, 4))

sns.boxplot(x=df_clean['distance_km'])

plt.title('Boxplot of Delivery Distance')
plt.xlabel('Delivery Distance (km)')

plt.show()


Analyse the relationship between delivery distance and delivery time

In [ ]:
# Select rows where both distance and delivery time are available
distance_time = df_clean[
    ['distance_km', 'Time_taken (min)']
].dropna()

# Calculate correlation
correlation = distance_time['distance_km'].corr(
    distance_time['Time_taken (min)']
)

print("Correlation between distance and delivery time:", correlation)

# Create scatter plot with a trend line
plt.figure(figsize=(9, 5))

sns.regplot(
    data=distance_time,
    x='distance_km',
    y='Time_taken (min)',
    scatter_kws={'alpha': 0.2},
    line_kws={'color': 'red'}
)

plt.title('Relationship Between Delivery Distance and Delivery Time')
plt.xlabel('Delivery Distance (km)')
plt.ylabel('Delivery Time (minutes)')
plt.show()

Delivery Time Analysis Across Operational Factors

In [ ]:

# Average delivery time by road traffic density

traffic_time = (
    df_clean.groupby('Road_traffic_density')['Time_taken (min)']
    .mean()
    .sort_values(ascending=False)
)

print("Average Delivery Time by Traffic Density:")
print(traffic_time.round(2))

plt.figure(figsize=(8, 5))
sns.barplot(
    x=traffic_time.index,
    y=traffic_time.values
)

plt.title('Delivery Time Varies Across Traffic Conditions')
plt.xlabel('Road Traffic Density')
plt.ylabel('Average Delivery Time (minutes)')
plt.show()


# Average delivery time by weather conditions

weather_time = (
    df_clean.groupby('Weather_conditions')['Time_taken (min)']
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Delivery Time by Weather:")
print(weather_time.round(2))

plt.figure(figsize=(9, 5))
sns.barplot(
    x=weather_time.index,
    y=weather_time.values
)

plt.title('Weather Conditions and Average Delivery Time')
plt.xlabel('Weather Conditions')
plt.ylabel('Average Delivery Time (minutes)')
plt.xticks(rotation=30)
plt.show()


# Average delivery time by vehicle type

vehicle_time = (
    df_clean.groupby('Type_of_vehicle')['Time_taken (min)']
    .mean()
    .sort_values(ascending=False)
)

print("\nAverage Delivery Time by Vehicle:")
print(vehicle_time.round(2))

plt.figure(figsize=(8, 5))
sns.barplot(
    x=vehicle_time.index,
    y=vehicle_time.values
)

plt.title('Average Delivery Time Across Vehicle Types')
plt.xlabel('Vehicle Type')
plt.ylabel('Average Delivery Time (minutes)')
plt.show()


My observation:
I compared the average delivery time across different traffic conditions, weather conditions and vehicle types. This helped me understand how operational factors may be associated with delivery delays. I observed that delivery time is not influenced by just one factor. Traffic, weather and the type of vehicle may all contribute to differences in delivery time. I will examine these relationships further through regression analysis.

Correlation heatmap

In [ ]:
# Select numerical variables for correlation analysis
numeric_columns = [
    'Delivery_person_Age',
    'Delivery_person_Ratings',
    'Vehicle_condition',
    'multiple_deliveries',
    'distance_km',
    'Time_taken (min)'
]

# Calculate correlation
correlation_matrix = df_clean[numeric_columns].corr()

# Display correlation values
print("Correlation Matrix:")
print(correlation_matrix.round(2))

# Create heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title('Correlation Between Delivery Variables')
plt.show()

My observation:
The correlation analysis shows a positive relationship between delivery distance and delivery time, with a correlation value of approximately 0.32. This suggests that longer delivery distances tend to be associated with longer delivery times. However, the relationship is not very strong, which indicates that other factors may also influence delivery time. Correlation does not establish causation.

Prepare the dataset for regression

In [ ]:
# Select variables required for regression
regression_columns = [
    'Time_taken (min)',
    'Delivery_person_Age',
    'Delivery_person_Ratings',
    'distance_km',
    'Road_traffic_density',
    'Weather_conditions',
    'Type_of_vehicle',
    'City',
    'Festival',
    'multiple_deliveries'
]

# Create a separate regression dataset
regression_data = df_clean[regression_columns].copy()

# Remove records where any selected variable is missing
regression_data = regression_data.dropna()

# Convert categorical variables into category data type
categorical_columns = [
    'Road_traffic_density',
    'Weather_conditions',
    'Type_of_vehicle',
    'City',
    'Festival',
    'multiple_deliveries'
]

for col in categorical_columns:
    regression_data[col] = regression_data[col].astype('category')

# Check the prepared dataset
print("Regression Dataset Shape:", regression_data.shape)

print("\nMissing Values:")
print(regression_data.isnull().sum())

print("\nData Types:")
print(regression_data.dtypes)

print("\nFirst Five Rows:")
display(regression_data.head())

My observation:
For regression analysis, I selected delivery time as my dependent variable and delivery distance, delivery-person age, ratings, traffic, weather, vehicle type, city, festival and multiple deliveries as independent variables. I excluded records where the required delivery distance was unavailable because of unreliable GPS coordinates. Therefore, the regression analysis uses a smaller sample than the original dataset. This helps me avoid using potentially incorrect distance values in the model.

Model 1 — Multiple Linear Regression without interaction

In [ ]:
# Import the regression library
import statsmodels.formula.api as smf

# Model 1: Multiple Linear Regression without interaction
model1 = smf.ols(
    formula="""
    Q('Time_taken (min)') ~
    Delivery_person_Age +
    Delivery_person_Ratings +
    distance_km +
    C(Road_traffic_density, Treatment(reference='High')) +
    C(Weather_conditions, Treatment(reference='Cloudy')) +
    C(Type_of_vehicle, Treatment(reference='motorcycle')) +
    C(City, Treatment(reference='Metropolitian')) +
    C(Festival, Treatment(reference='No')) +
    C(multiple_deliveries, Treatment(reference=0))
    """,
    data=regression_data
).fit()

# Display the complete regression results
print(model1.summary())

Model 2 — Multiple Linear Regression with interaction

In [ ]:
# Model 2: Multiple Linear Regression with interaction

model2 = smf.ols(
    formula="""
    Q('Time_taken (min)') ~
    Delivery_person_Age +
    Delivery_person_Ratings +
    distance_km +
    C(Road_traffic_density, Treatment(reference='High')) *
    C(multiple_deliveries, Treatment(reference=0)) +
    C(Weather_conditions, Treatment(reference='Cloudy')) +
    C(Type_of_vehicle, Treatment(reference='motorcycle')) +
    C(City, Treatment(reference='Metropolitian')) +
    C(Festival, Treatment(reference='No'))
    """,
    data=regression_data
).fit()

# Display complete regression results
print(model2.summary())

Compare Model 1 and Model 2

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# Calculate prediction errors for Model 1
pred1 = model1.predict(regression_data)
rmse1 = np.sqrt(mean_squared_error(
    regression_data['Time_taken (min)'], pred1
))
mae1 = mean_absolute_error(
    regression_data['Time_taken (min)'], pred1
)

# Calculate prediction errors for Model 2
pred2 = model2.predict(regression_data)
rmse2 = np.sqrt(mean_squared_error(
    regression_data['Time_taken (min)'], pred2
))
mae2 = mean_absolute_error(
    regression_data['Time_taken (min)'], pred2
)

# Count statistically significant predictors (excluding intercept)
sig1 = (model1.pvalues.drop('Intercept') < 0.05).sum()
sig2 = (model2.pvalues.drop('Intercept') < 0.05).sum()

# Create model comparison table
model_comparison = pd.DataFrame({
    'Measure': [
        'R-squared',
        'Adjusted R-squared',
        'AIC',
        'BIC',
        'F-statistic',
        'F-statistic p-value',
        'RMSE (minutes)',
        'MAE (minutes)',
        'Significant predictors'
    ],
    'Model 1': [
        model1.rsquared,
        model1.rsquared_adj,
        model1.aic,
        model1.bic,
        model1.fvalue,
        model1.f_pvalue,
        rmse1,
        mae1,
        sig1
    ],
    'Model 2': [
        model2.rsquared,
        model2.rsquared_adj,
        model2.aic,
        model2.bic,
        model2.fvalue,
        model2.f_pvalue,
        rmse2,
        mae2,
        sig2
    ]
})

print("MODEL COMPARISON")
display(model_comparison.round(4))

My observation:
I compared two multiple linear regression models. Model 1 does not include an interaction effect, whereas Model 2 examines whether the relationship between traffic density and delivery time changes when delivery partners handle multiple orders.

Model 2 has a slightly higher R-squared value of 0.5814 compared with 0.5805 for Model 1. Its adjusted R-squared is also slightly higher. Model 2 has a lower AIC, RMSE and MAE, while Model 1 has a lower BIC.

The improvement in Model 2 is very small. Based on the overall comparison, Model 2 provides a marginal improvement in prediction, but the difference is not substantial.

Residual analysis

In [ ]:
# Calculate residuals for Model 2
residuals = model2.resid
predicted_values = model2.fittedvalues

# Residuals vs predicted values
plt.figure(figsize=(9, 5))

sns.scatterplot(
    x=predicted_values,
    y=residuals,
    alpha=0.2
)

plt.axhline(y=0, color='red', linestyle='--')

plt.title('Residuals vs Predicted Delivery Time')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Residuals (minutes)')

plt.show()

In [ ]:
# Histogram of residuals for Model 2

plt.figure(figsize=(8, 5))

sns.histplot(
    residuals,
    bins=30,
    kde=True
)

plt.axvline(x=0, color='red', linestyle='--')

plt.title('Distribution of Regression Residuals')
plt.xlabel('Residuals (minutes)')
plt.ylabel('Frequency')

plt.show()

My observation:
I used residual analysis to examine how well my regression model fits the data. The residual-versus-predicted plot helps me identify whether errors follow a visible pattern. The histogram helps me understand the distribution of prediction errors. Ideally, residuals should be centred around zero without a clear pattern. Any visible pattern or strong skewness may indicate that the model does not capture all the factors affecting delivery time.

Analyse the regression coefficients

In [ ]:
# Extract Model 2 coefficients
coefficient_table = pd.DataFrame({
    'Coefficient': model2.params,
    'P-value': model2.pvalues
})

# Add significance status
coefficient_table['Significance'] = np.where(
    coefficient_table['P-value'] < 0.05,
    'Significant',
    'Not Significant'
)

# Display results
display(coefficient_table.round(4))

My interpretation of the regression results:
1. Delivery distance: The coefficient indicates how much predicted delivery time changes when delivery distance increases by one kilometre, keeping the other variables constant.
2. Delivery-person ratings: The coefficient indicates how predicted delivery time changes when the delivery-person rating increases by one point, keeping the other variables constant.
3. Multiple deliveries / traffic: The categorical coefficients show how predicted delivery time differs from the selected reference category, after controlling for the other variables.

In [ ]:
# Sort coefficients, excluding the intercept
coefficient_table_without_intercept = (
    coefficient_table.drop(index='Intercept')
)

print("Highest Positive Coefficients:")
display(
    coefficient_table_without_intercept
    .sort_values('Coefficient', ascending=False)
    .head(5)
)

print("Most Negative Coefficients:")
display(
    coefficient_table_without_intercept
    .sort_values('Coefficient')
    .head(5)
)

Business interpretation and recommendations

In [ ]:
# Average delivery time by traffic density
print("Delivery Time by Traffic:")
display(
    df_clean.groupby('Road_traffic_density')['Time_taken (min)']
    .agg(['mean', 'median', 'count'])
    .round(2)
)

# Average delivery time by weather
print("\nDelivery Time by Weather:")
display(
    df_clean.groupby('Weather_conditions')['Time_taken (min)']
    .agg(['mean', 'median', 'count'])
    .round(2)
)

# Average delivery time by multiple deliveries
print("\nDelivery Time by Multiple Deliveries:")
display(
    df_clean.groupby('multiple_deliveries')['Time_taken (min)']
    .agg(['mean', 'median', 'count'])
    .round(2)
)

Business recommendations:

Based on my analysis, I recommend the following actions:
1. Traffic management: Delivery operations should consider traffic conditions while estimating delivery times and assigning orders. This can help improve the accuracy of estimated delivery times.
2. Multiple-order allocation: The company should review how many orders are assigned to one delivery partner at a time. Multiple-order assignments should be managed carefully, particularly when traffic conditions are challenging.
3. Weather-based planning: Delivery operations should consider weather conditions when planning delivery estimates and allocating delivery partners. This may help reduce delays during difficult weather conditions.

Check the interaction effect

In [ ]:
# Display only interaction-related coefficients
interaction_results = coefficient_table[
    coefficient_table.index.str.contains(':', regex=False)
]

print("Traffic and Multiple Deliveries Interaction Results:")
display(interaction_results.round(4))

My observation: I included an interaction between road traffic density and the number of multiple deliveries to examine whether the effect of traffic on delivery time changes when a delivery partner handles more than one order. I checked the interaction coefficients and their p-values to understand whether these combined effects are statistically significant. This helps me examine whether traffic and multiple deliveries should be studied separately or together.

In [ ]:
# Display the interaction terms and their p-values
for term, row in interaction_results.iterrows():
    print("\nInteraction:", term)
    print("Coefficient:", round(row['Coefficient'], 4))
    print("P-value:", round(row['P-value'], 4))

    if row['P-value'] < 0.05:
        print("Interpretation: This interaction is statistically significant.")
    else:
        print("Interpretation: This interaction is not statistically significant.")

In [ ]:
# Export model comparison table
model_comparison.to_csv(
    'zomato_model_comparison.csv',
    index=False
)

# Export Model 2 coefficient results
coefficient_table.to_csv(
    'zomato_model2_coefficients.csv'
)

# Export descriptive delivery-time summaries
traffic_time = (
    df_clean.groupby('Road_traffic_density')['Time_taken (min)']
    .agg(['mean', 'median', 'count'])
)

weather_time = (
    df_clean.groupby('Weather_conditions')['Time_taken (min)']
    .agg(['mean', 'median', 'count'])
)

multiple_delivery_time = (
    df_clean.groupby('multiple_deliveries')['Time_taken (min)']
    .agg(['mean', 'median', 'count'])
)

traffic_time.to_csv('traffic_delivery_summary.csv')
weather_time.to_csv('weather_delivery_summary.csv')
multiple_delivery_time.to_csv('multiple_delivery_summary.csv')

print("All analysis files exported successfully.")

Limitations of my analysis:
1. The dataset contains missing values in some variables, so I used median imputation for selected numerical variables and the category "Unknown" for selected categorical variables.
2. Some GPS coordinates appeared to be unreliable. I excluded records with flagged GPS coordinates from the distance-based regression analysis.
3. The dataset represents a limited period and may not capture all changes in delivery operations over time.
4. The regression model explains only part of the variation in delivery time. Other factors that are not included in the dataset may also affect delivery performance.
5. The analysis identifies associations between variables and delivery time; it does not establish cause-and-effect relationships.